# EDA — expense_data_1.csv

**Objetivo:** determinar calidad, significado y suficiencia temporal para APERTURE. Análisis descriptivo, sin entrenar modelos. Los resultados textuales siguientes corresponden al archivo cuyo hash se imprime; si cambia la fuente, revisar las interpretaciones.

**Entorno:** seleccionar `.venv` del repositorio e instalar `requirements-eda.txt`. El notebook utiliza fechas originales, no fechas desplazadas para la demo.

In [1]:
from pathlib import Path
import sys, json, hashlib
import pandas as pd
import plotly.express as px
import plotly.io as pio
from IPython.display import display
from decimal import Decimal
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'scripts/eda_datasets.py').exists())
sys.path.insert(0,str(ROOT))
from scripts.eda_datasets import audit
paths=sorted((ROOT/'data/demo').rglob('expense_data_1.csv'))
assert len(paths)==1, 'Archivo ausente o ambiguo'
path=paths[0]
raw=pd.read_csv(path,dtype=str) if path.suffix=='.csv' else pd.read_excel(path)
summary,missing,monthly,weekly,categories,weekly_categories=audit(path)
pio.renderers.default='plotly_mimetype'
print('Archivo:',path.name,'SHA-256:',summary['sha256'])
print('No se escribirá en el original ni en la base de la aplicación.')


Archivo: expense_data_1.csv SHA-256: b7e5cc0bca39f51fe6de8ae64a95edd78e71bdecb88952860385b10ea092e090
No se escribirá en el original ni en la base de la aplicación.


## A. Estructura y ejemplos

**Objetivo:** conocer la unidad de registro, tipos, memoria y posibles identificadores. Los ejemplos omiten cuentas y notas para reducir exposición de información financiera.

In [2]:
print('Filas y columnas:',raw.shape)
print('Memoria profunda (bytes):',raw.memory_usage(deep=True).sum())
display(pd.DataFrame({'tipo':raw.dtypes.astype(str),'valores_distintos':raw.nunique()}))
display(raw[['Date','Category','Amount']].head())
print('Etiquetas de cuenta distintas:',summary['account_labels_count'])
print('No hay identificador explícito de persona; varias cuentas no implican varias personas.')


Filas y columnas: (277, 11)
Memoria profunda (bytes): 179347


,tipo,valores_distintos
Date,object,240
Account,object,2
Category,object,13
Subcategory,object,0
Note,object,159
INR,object,138
Income/Expense,object,2
Note.1,object,0
Amount,object,137
Currency,object,2


,Date,Category,Amount
0,3/2/2022 10:11,Food,50
1,3/2/2022 10:11,Other,300
2,3/1/2022 19:50,Food,78
3,3/1/2022 18:56,Transportation,30
4,3/1/2022 18:22,Food,67


Etiquetas de cuenta distintas: 2
No hay identificador explícito de persona; varias cuentas no implican varias personas.


**Observado:** 277 filas, 11 columnas, 2 etiquetas de cuenta; memoria profunda 179,347 bytes en el entorno auditado.

**Interpretación:** cada fila representa un movimiento según la fuente, no una observación semanal independiente.

**Implicación:** conservar cuentas y procedencia, sin construir usuarios a partir de nombres de cuenta.

## B. Calidad, faltantes y duplicados

**Objetivo:** distinguir errores confirmados, campos opcionales vacíos y señales que requieren revisión.

In [3]:
display(missing)
print({k:summary[k] for k in ['invalid_dates','invalid_amounts','nonpositive_amounts','exact_duplicate_rows','coarse_duplicate_candidates']})
print('Descripciones ambiguas candidatas:',summary['ambiguous_note_candidates'])
print('Espacios en categorías:',summary['category_whitespace_rows'])
display(px.bar(missing,x='column',y='fraction',title='Fracción faltante por columna'))


,column,missing,fraction
0,Date,0,0.00000
1,Account,0,0.00000
2,Category,0,0.00000
3,Subcategory,277,1.00000
4,Note,4,0.01444
5,INR,0,0.00000
6,Income/Expense,0,0.00000
7,Note.1,277,1.00000
8,Amount,0,0.00000
9,Currency,0,0.00000


{'invalid_dates': 0, 'invalid_amounts': 0, 'nonpositive_amounts': 0, 'exact_duplicate_rows': 0, 'coarse_duplicate_candidates': 23}
Descripciones ambiguas candidatas: 61
Espacios en categorías: 0


**Observado:** fechas e importes válidos y positivos en las filas auditadas; cero duplicados exactos. Hay 23 filas que coinciden en una clave amplia. Columnas completamente vacías: ['Subcategory', 'Note.1'].

**Interpretación:** una coincidencia de fecha, importe, tipo y cuenta no prueba duplicación; un importe grande puede ser legítimo.

**Implicación:** conservar filas y agregar banderas; no borrar ni imputar automáticamente.

## C. Cobertura temporal y flujos

**Objetivo:** estudiar periodos, frecuencia e ingresos/gastos registrados. Los valores se expresan en unidades menores de la unidad de análisis; no son COP salvo conversión explícita. Las semanas son lunes-domingo.

In [4]:
print('Periodo:',summary['date_min'],'a',summary['date_max'])
print('Semanas dentro de límites de fechas:',summary['calendar_bounded_weeks'])
print('Unidad de análisis:',summary['analysis_unit'])
display(monthly)
m=monthly.reset_index(names='mes');m['mes']=m.mes.astype(str)
display(px.bar(m,x='mes',y=['inflow_minor','expense_minor'],barmode='group',title='Entradas y gastos registrados por mes (unidades menores)'))
w=weekly.reset_index(names='semana');w['semana']=w.semana.astype(str)
display(px.line(w,x='semana',y='expense_minor',color='inside_date_bounds',title='Gastos semanales; bordes parciales diferenciados'))


Periodo: 2021-11-21 a 2022-03-02
Semanas dentro de límites de fechas: 14
Unidad de análisis: INR: columna de equivalente documentada


,transactions,days_with_records,expense_minor,inflow_minor,recorded_cashflow_minor,inside_date_bounds
2021-11,35,10,362040,400300,38260,False
2021-12,89,25,1633444,1231000,-402444,True
2022-01,92,27,2067951,1964100,-103851,True
2022-02,54,23,1786465,1820000,33535,True
2022-03,7,2,52500,60000,7500,False


**Observado:** 2021-11-21 a 2022-03-02, 5 meses tocados y 14 semanas enteras dentro de los límites. No se observaron semanas interiores totalmente vacías.

**Interpretación:** el límite de fechas es solo una cota calendárica; no certifica que existan todas las transacciones. Ausencia de registros no equivale a gasto cero.

**Implicación:** revisar bordes y cobertura antes del experimento; no desplazar fechas para evaluar ni consultar tasas.

## D. Categorías, dispersión y valores extremos

**Objetivo:** estudiar frecuencia, importe, media/mediana y actividad semanal, además de señales IQR. Los valores extremos se conservan.

In [5]:
display(pd.Series(summary['category_counts'],name='frecuencia').to_frame())
display(categories)
top=list(categories.head(5).category)
z=weekly_categories[top].reset_index(names='semana');z['semana']=z.semana.astype(str)
display(px.line(z,x='semana',y=top,title='Evolución por categoría (unidades menores)'))
amount_col='INR' if 'INR' in raw else 'Amount'
x=pd.to_numeric(raw[amount_col],errors='coerce')
display(px.histogram(x=x,nbins=35,title='Distribución de importes: gráfico exploratorio, sin eliminar extremos'))


,frecuencia
Food,156
Other,60
Transportation,31
Apparel,7
Household,6
Allowance,6
Social Life,5
Education,1
Salary,1
Self-development,1


,category,transactions,total_minor,median_transaction_minor,mean_transaction_decimal,active_bounded_weeks,fraction_zero_weeks,iqr_upper_flags
3,Food,156,2450248,9450,157.0671794871794871794871795,14,0.000000,11.0
5,Household,6,1218800,121750,2031.333333333333333333333333,4,0.714286,NaN
9,Transportation,31,920380,13000,296.8967741935483870967741935,10,0.285714,5.0
6,Other,22,511700,10000,232.5909090909090909090909091,10,0.285714,1.0
0,Apparel,7,338800,39900,484,3,0.785714,NaN
8,Social Life,5,251372,20000,502.744,5,0.642857,NaN
2,Education,1,140000,140000,1400,1,0.928571,NaN
7,Self-development,1,40000,40000,400,1,0.928571,NaN
1,Beauty,1,19600,19600,196,1,0.928571,NaN
4,Gift,1,11500,11500,115,1,0.928571,NaN


**Observado:** las categorías tienen frecuencias desiguales; véanse `active_bounded_weeks` y `fraction_zero_weeks`. El criterio IQR se aplica solo a grupos con al menos ocho registros.

**Interpretación:** categorías mensuales como vivienda pueden contener muchos ceros semanales. La media y la mediana describen aspectos diferentes de una distribución sesgada.

**Implicación:** seleccionar categorías con suficiente actividad únicamente dentro de entrenamiento; no confundir una bandera IQR con error o fraude.

## E. Limpieza semántica y monedas

**Objetivo:** documentar las transformaciones permitidas y las decisiones pendientes.

In [6]:
print('Monedas:',summary['currency_counts'])
print('Pagos de tarjeta:',summary['card_payment_rows'])
print('Patas sin pareja con coincidencia estricta:',summary['unpaired_card_legs_coarse_match'])
print('Diferencias entre Amount e INR:',summary['original_vs_inr_differences'])
print('Account.1 igual a Amount:',summary['amount_account1_equal'])


Monedas: {'INR': 276, 'USD': 1}
Pagos de tarjeta: 0
Patas sin pareja con coincidencia estricta: 0
Diferencias entre Amount e INR: 1
Account.1 igual a Amount: True


**Observado:** 276 operaciones INR y una USD; Amount y Account.1 coinciden. Dos columnas están totalmente vacías y Note tiene cuatro faltantes.

**Interpretación:** INR es el equivalente documentado de la fuente; no sustituye el importe original. Las notas From/To no demuestran que exista un préstamo.

**Implicación:** excluir columnas vacías solo de tablas derivadas, conservar notas opcionales y revisar tipos ambiguos. Para COP usar cada moneda original y su fecha original.

## F. Dictamen y reproducibilidad

**Objetivo:** decidir usos viables sin inventar resultados predictivos.

In [7]:
print('El original permanece intacto:',hashlib.sha256(path.read_bytes()).hexdigest()==summary['sha256'])
print('Modelos entrenados en este notebook: ninguno')


El original permanece intacto: True
Modelos entrenados en este notebook: ninguno


**Dictamen:** Útil para probar importación y multimoneda. Catorce semanas interiores constituyen muy poco historial para comparar rigurosamente una red neuronal con múltiples categorías.

**Siguiente paso:** revisión humana de tipos dudosos, confirmación de cobertura y validación temporal frente a última semana/promedio móvil. Los sintéticos se reservan para desarrollo y pruebas.

Fuentes: [Kaggle Personal Finance](https://www.kaggle.com/datasets/entrepreneurlife/personal-finance/data), [Kaggle My Expenses](https://www.kaggle.com/datasets/tharunprabu/my-expenses-data), [Frankfurter](https://frankfurter.dev/).